# Stage 13D — Historical Landslide Coordinate Recovery and Evidence Audit

**Purpose:** audit all coordinate-ineligible Malaysian historical landslides and prepare an evidence-based manual correction workflow. This notebook **does not geocode place names to invented point coordinates**, download articles, rewrite the master inventory, or retrain models. The previously trained 194-case static and 121-case temporal models remain unchanged.

**Recovery acceptance rule:** a new point needs a source-specific, independently verifiable coordinate or identifiable landslide footprint, unambiguous event identity, documented location uncertainty, and appropriate resolution for 30-m geospatial features. A district/town centroid is never an acceptable substitute.

## PART A — Paths, inventory structure and safety checks

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"
OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13d_coordinate_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWED_PATH = DEV_DIR / "02_cleaned_inventory" / "malaysia_positive_inventory_final.csv"
ELIGIBLE_PATH = DEV_DIR / "02_cleaned_inventory" / "malaysia_positive_inventory_ml_eligible.csv"
STAGE04_PATH = DEV_DIR / "01_preprocessing" / "stage04_revised_outputs" / "positive_inventory_with_ml_eligibility.csv"

for label, path in [('reviewed', REVIEWED_PATH), ('ml_eligible', ELIGIBLE_PATH), ('stage04', STAGE04_PATH)]:
    print(f'{label:14s}', 'FOUND' if path.exists() else 'MISSING', path)
if not REVIEWED_PATH.exists() or not ELIGIBLE_PATH.exists():
    raise FileNotFoundError('Required inventory files missing. Update REVIEWED_PATH / ELIGIBLE_PATH in PART A.')

reviewed = pd.read_csv(REVIEWED_PATH, low_memory=False)
eligible = pd.read_csv(ELIGIBLE_PATH, low_memory=False)
stage04 = pd.read_csv(STAGE04_PATH, low_memory=False) if STAGE04_PATH.exists() else pd.DataFrame()

for name, df in [('Reviewed', reviewed), ('Eligible', eligible), ('Stage04', stage04)]:
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    print('Columns:', ', '.join(map(str, df.columns)))

# Review flags rather than silently changing identifiers or dropping records.
if len(reviewed) != 287 or len(eligible) != 200:
    print('NOTE: counts differ from previous 287 / 200; inspect lineage before interpreting exclusions.')

reviewed       FOUND C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.csv
ml_eligible    FOUND C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.csv
stage04        FOUND C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\stage04_revised_outputs\positive_inventory_with_ml_eligibility.csv
Reviewed: 287 rows, 19 columns
Columns: final_positive_id, positive_id, source_dataset, source_record_id, source_event_id, event_date, event_date_available, event_date_source_reliable_for_review, state_or_admin, district_or_locality, event_title_or_name, location_accuracy_source, longitude, latitude, dedup_status, duplicate_group, representative_record_id, duplicate_reason, inside_malaysia_boundary
Eligible: 200 rows, 29 columns
Columns: final_positive_id, positive_id, source_dataset, source_record_id, source_event_id, event_date, event_date_available, event_date_source_reliable_for_review, st

## PART B — Identify excluded records without unreliable coordinate-only matching

In [2]:
def select_key(left, right):
    # Require same ID name, completeness, and uniqueness on BOTH sides.
    priority = ['sample_id', 'positive_id', 'landslide_id', 'inventory_id',
                'event_id', 'record_id', 'source_id', 'id']
    for field in priority:
        if field in left and field in right:
            a = left[field].astype('string').str.strip()
            z = right[field].astype('string').str.strip()
            if a.notna().all() and z.notna().all() and a.is_unique and z.is_unique:
                overlap = len(set(a) & set(z))
                if overlap >= min(len(right), len(left)) * 0.90:
                    return field, overlap
    return None, 0

KEY, overlap = select_key(reviewed, eligible)
print('Chosen audited inventory key:', KEY, 'overlap:', overlap)
if KEY is None:
    pd.DataFrame({'dataset':['reviewed','eligible'],
                  'columns':[','.join(reviewed.columns),','.join(eligible.columns)]}).to_csv(
                      OUTPUT_DIR / 'identifier_manual_review_required.csv', index=False)
    raise ValueError('No sufficiently complete matching unique ID. Inspect identifier_manual_review_required.csv. Do NOT match by rounded coordinates.')

reviewed = reviewed.copy()
eligible = eligible.copy()
reviewed['_audit_key'] = reviewed[KEY].astype('string').str.strip()
eligible['_audit_key'] = eligible[KEY].astype('string').str.strip()
known_eligible = set(eligible['_audit_key'])
excluded = reviewed.loc[~reviewed['_audit_key'].isin(known_eligible)].copy()
print('Excluded records identified:',len(excluded))

# Link Stage04 by the SAME ID only where Stage04 IDs uniquely agree.
if not stage04.empty and KEY in stage04.columns:
    tmp = stage04.copy()
    tmp['_audit_key'] = tmp[KEY].astype('string').str.strip()
    if tmp['_audit_key'].notna().all() and tmp['_audit_key'].is_unique:
        extra = [col for col in tmp.columns if col not in reviewed.columns and col != '_audit_key']
        excluded = excluded.merge(tmp[['_audit_key'] + extra], on='_audit_key', how='left', validate='one_to_one')
        print('Joined additional Stage04 columns:',extra)
    else:
        print('WARNING: Stage04 key not unique/complete. Skipping automated Stage04 join.')
else:
    print('NOTE: Stage04 absent or incompatible ID; eligibility reason may require manual checking.')

excluded.to_csv(OUTPUT_DIR / 'coordinate_ineligible_source_rows.csv', index=False)
assert excluded['_audit_key'].is_unique

Chosen audited inventory key: positive_id overlap: 200
Excluded records identified: 87
Joined additional Stage04 columns: ['location_accuracy_m', 'ml_eligible', 'ml_exclusion_reason', 'event_date_parsed', 'event_year', 'event_month', 'event_day', 'date_precision_for_modelling', 'annual_landcover_eligible', 'antecedent_rainfall_eligible']


## PART C — Inspect original coordinate quality and source evidence

In [3]:
def cols_with(patterns, frame):
    return [col for col in frame.columns if any(p in col.lower() for p in patterns)]

fields = cols_with(['accur','precis','uncertain','reason','eligib','latitude','longitude',
                    'lat','lon','location','description','source','url','link','date',
                    'state','district','jmg','coolr','country'], excluded)
print('Potential relevant fields:')
print(fields)

for field in cols_with(['accur','precis','uncertain','reason','eligib'], excluded):
    print('\n',field)
    print(excluded[field].astype('string').fillna('<MISSING>').value_counts().head(15))

# Preserve all source columns; append triage metadata rather than rewriting raw data.
source_fields = cols_with(['source','url','link','location','descrip','date','state','district','accur','precis'], excluded)
triage = excluded[['_audit_key'] + [f for f in source_fields if f != '_audit_key']].copy()

def meaningful(value):
    if pd.isna(value): return False
    value = str(value).strip().lower()
    return value not in ('','nan','none','null','unknown','n/a','not available')

source_link_cols = cols_with(['url','link'], triage)
location_cols = cols_with(['location','descrip','place','district'], triage)
triage['has_source_link'] = triage[source_link_cols].apply(lambda r:any(map(meaningful,r)),axis=1) if source_link_cols else False
triage['has_location_description'] = triage[location_cols].apply(lambda r:any(map(meaningful,r)),axis=1) if location_cols else False
triage['review_priority'] = np.select(
    [triage['has_source_link'] & triage['has_location_description'],
     triage['has_source_link'] | triage['has_location_description']],
    ['A_source_and_description','B_partial_evidence'], default='C_source_search_needed')
triage['evidence_review_status'] = 'NOT_REVIEWED'
triage['proposed_latitude'] = pd.NA
triage['proposed_longitude'] = pd.NA
triage['proposed_accuracy_m'] = pd.NA
triage['proposed_source_url_or_reference'] = pd.NA
triage['proposed_evidence_type'] = pd.NA
triage['event_identity_verified'] = False
triage['location_evidence_verified'] = False
triage['reviewer_notes'] = pd.NA
triage = triage.sort_values(['review_priority','_audit_key']).reset_index(drop=True)
triage.to_csv(OUTPUT_DIR / 'coordinate_source_review_queue.csv',index=False)
print(triage['review_priority'].value_counts())
print('Review queue saved. Human review should use this queue only as supporting evidence.')
display(triage.head(12))

Potential relevant fields:
['source_dataset', 'source_record_id', 'source_event_id', 'event_date', 'event_date_available', 'event_date_source_reliable_for_review', 'state_or_admin', 'district_or_locality', 'location_accuracy_source', 'longitude', 'latitude', 'duplicate_reason', 'location_accuracy_m', 'ml_eligible', 'ml_exclusion_reason', 'event_date_parsed', 'date_precision_for_modelling', 'annual_landcover_eligible', 'antecedent_rainfall_eligible']

 location_accuracy_source
location_accuracy_source
25km       34
10km       31
50km       15
unknown     6
250km       1
Name: count, dtype: int64[pyarrow]

 duplicate_reason
duplicate_reason
<MISSING>                                                      86
Same coordinate + same event date + matching title/location     1
Name: count, dtype: int64[pyarrow]

 location_accuracy_m
location_accuracy_m
25000.0      34
10000.0      31
50000.0      15
<MISSING>     6
250000.0      1
Name: count, dtype: int64[pyarrow]

 ml_eligible
ml_eligible
Fal

,_audit_key,source_dataset,source_record_id,source_event_id,event_date,event_date_available,event_date_source_reliable_for_review,state_or_admin,district_or_locality,location_accuracy_source,...,review_priority,evidence_review_status,proposed_latitude,proposed_longitude,proposed_accuracy_m,proposed_source_url_or_reference,proposed_evidence_type,event_identity_verified,location_evidence_verified,reviewer_notes
0,COOLR_11025,NASA_COOLR,COOLR_11025,11025,2017-04-12 20:00:00,True,True,Perak,"Gerik, Hulu Perak, Perak, Malaysia",25km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
1,COOLR_11026,NASA_COOLR,COOLR_11026,11026,2017-01-27 00:00:00,True,True,Perak,"4, Temenggor, Hulu Perak, 33320, Perak",25km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
2,COOLR_11028,NASA_COOLR,COOLR_11028,11028,2017-01-02 16:00:00,True,True,Kelantan,"185, Bertam, Gua Musang, 18300, Kelantan",25km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
3,COOLR_11029,NASA_COOLR,COOLR_11029,11029,2016-12-10 00:00:00,True,True,Pahang,"Forest, Matau Gelangi, Jerantut",50km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
4,COOLR_11035,NASA_COOLR,COOLR_11035,11035,2017-01-24 00:00:00,True,True,Pahang,"9, Sabai, Bentong, 28620, Pahang",10km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
5,COOLR_11044,NASA_COOLR,COOLR_11044,11044,2017-04-01 00:00:00,True,True,Sabah,"Kinabatangan, Division Sandakan, Sabah, Malaysia",25km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
6,COOLR_11046,NASA_COOLR,COOLR_11046,11046,2017-05-11 00:00:00,True,True,Sarawak,"Long Panai-Marudi road, Division Miri, Sarawak...",10km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
7,COOLR_11047,NASA_COOLR,COOLR_11047,11047,2017-02-22 06:00:00,True,True,Sarawak,"Jalan Kuala Tatau, Tatau, Division Bintulu, 97...",10km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
8,COOLR_11048,NASA_COOLR,COOLR_11048,11048,2017-03-20 18:00:00,True,True,Sarawak,"Rumah Empaling, Sg Labang, near SK Sengian Lab...",10km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>
9,COOLR_11050,NASA_COOLR,COOLR_11050,11050,2017-03-18 00:00:00,True,True,Sarawak,"Kg Rian, Song district, Kapit Division, Sarawa...",50km,...,B_partial_evidence,NOT_REVIEWED,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,<NA>


## PART D — Manual evidence template and verification of proposed corrections

In [4]:
MANUAL_PATH = OUTPUT_DIR / 'manual_coordinate_evidence_entries.csv'
if not MANUAL_PATH.exists():
    template = pd.DataFrame({
        '_audit_key': excluded['_audit_key'].astype('string'),
        'proposed_latitude': pd.Series(dtype='float64'),
        'proposed_longitude': pd.Series(dtype='float64'),
        'proposed_accuracy_m': pd.Series(dtype='float64'),
        'proposed_source_url_or_reference': pd.Series(dtype='string'),
        'proposed_evidence_type': pd.Series(dtype='string'),
        'event_identity_verified': False,
        'location_evidence_verified': False,
        'reviewer_notes': pd.Series(dtype='string'),
    })
    template.to_csv(MANUAL_PATH,index=False)
    print('Created editable template:', MANUAL_PATH)
else:
    print('Preserving existing user-edited evidence file:',MANUAL_PATH)

entries = pd.read_csv(MANUAL_PATH, low_memory=False)
if '_audit_key' not in entries: raise KeyError('Evidence template must include _audit_key')
entries['_audit_key'] = entries['_audit_key'].astype('string').str.strip()
if entries['_audit_key'].duplicated().any():
    raise ValueError('Multiple proposals for the same event. Resolve duplicates manually, do not automatically choose.')
unknown_keys = set(entries['_audit_key'].dropna()) - set(excluded['_audit_key'])
if unknown_keys:
    raise ValueError(f'Proposals contain {len(unknown_keys)} IDs not present in excluded inventory.')

def is_true(value):
    return str(value).strip().lower() in ('true','1','yes','y')

entries['proposed_latitude'] = pd.to_numeric(entries['proposed_latitude'],errors='coerce')
entries['proposed_longitude'] = pd.to_numeric(entries['proposed_longitude'],errors='coerce')
entries['proposed_accuracy_m'] = pd.to_numeric(entries['proposed_accuracy_m'],errors='coerce')
entries['coordinates_valid'] = entries['proposed_latitude'].between(-90,90) & entries['proposed_longitude'].between(-180,180)
entries['within_broad_malaysia_bbox'] = entries['proposed_latitude'].between(0.5,7.7) & entries['proposed_longitude'].between(99,120.0)
entries['accuracy_meets_5km_rule'] = entries['proposed_accuracy_m'].gt(0) & entries['proposed_accuracy_m'].le(5000)
entries['source_provided'] = entries['proposed_source_url_or_reference'].map(meaningful)
entries['identity_verified'] = entries['event_identity_verified'].map(is_true)
entries['location_verified'] = entries['location_evidence_verified'].map(is_true)
entries['evidence_gate_pass'] = (entries['coordinates_valid'] & entries['within_broad_malaysia_bbox'] &
    entries['accuracy_meets_5km_rule'] & entries['source_provided'] & entries['identity_verified'] & entries['location_verified'])

# Bounding box is necessary but NOT proof of Malaysian land coverage. Use boundary geometry if installed.
entries['national_boundary_check'] = 'NOT_RUN'
try:
    import geopandas as gpd
    from shapely.geometry import Point
    boundary_path = PROJECT_ROOT / 'Landslide_FYP_Data' / '02_Boundary' / 'mys_admin0.geojson'
    if boundary_path.exists():
        boundary = gpd.read_file(boundary_path).to_crs('EPSG:4326')
        boundary_geom = boundary.geometry.union_all() if hasattr(boundary.geometry,'union_all') else boundary.unary_union
        entries['national_boundary_check'] = [
            'INSIDE' if (pd.notna(lo) and pd.notna(la) and boundary_geom.covers(Point(lo,la))) else 'OUTSIDE_OR_INVALID'
            for la,lo in zip(entries['proposed_latitude'],entries['proposed_longitude'])
        ]
    else:
        print('Malaysia polygon unavailable: national boundary check NOT_RUN')
except Exception as exc:
    print('National boundary check not completed:',type(exc).__name__,str(exc)[:250])

entries['automatic_validation_status'] = np.select(
    [entries['evidence_gate_pass'] & entries['national_boundary_check'].eq('INSIDE'),
     entries['evidence_gate_pass']],
    ['ELIGIBLE_FOR_INDEPENDENT_REVIEW', 'BOUNDARY_REVIEW_REQUIRED'],
    default='INSUFFICIENT_EVIDENCE')
entries.to_csv(OUTPUT_DIR/'proposed_coordinate_evidence_validation.csv',index=False)
print(entries['automatic_validation_status'].value_counts())
print('IMPORTANT: automatic checks are not independent source verification.')

Created editable template: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13d_coordinate_recovery\manual_coordinate_evidence_entries.csv
automatic_validation_status
INSUFFICIENT_EVIDENCE    87
Name: count, dtype: int64
IMPORTANT: automatic checks are not independent source verification.


## PART E — Recovery decision gate and next actions

In [5]:
summary = {
    'reviewed_inventory_count':len(reviewed),
    'original_ml_eligible_count':len(eligible),
    'coordinate_ineligible_count':len(excluded),
    'review_queue_with_links':int(triage['has_source_link'].sum()),
    'review_queue_with_descriptions':int(triage['has_location_description'].sum()),
    'independently_approved_coordinate_recoveries':0,
    'potentially_complete_manual_evidence_packages':int(entries['evidence_gate_pass'].sum()),
    'model_retraining_authorized':False,
    'current_static_positive_count':194,
    'current_temporal_positive_count':121,
    'note':'Manual and independent verification plus original Stage04 spatial/accuracy and full feature QC required before approval.'
}
with open(OUTPUT_DIR/'stage13d_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

checks = [
    ('Inventory lineage and matching','PASS' if KEY else 'BLOCKED',f'key={KEY}; excluded={len(excluded)}'),
    ('Original source evidence queue','PREPARED',f'{len(triage)} records for human review'),
    ('Proposed coordinate values','REVIEW_REQUIRED',f'{int(entries.evidence_gate_pass.sum())} pass preliminary evidence fields, not independently verified'),
    ('True Malaysia boundary','PASS' if (entries.national_boundary_check.eq('INSIDE').any()) else 'PENDING','Polygon check required for each proposed coordinate'),
    ('Independent evidence validation','PENDING','Manually open sources, verify event identity and coordinate precision'),
    ('Stage04 eligibility / feature QC rerun','PENDING','Rerun only for approved verified corrections'),
    ('LR / RF / XGBoost rebenchmark','NOT_AUTHORIZED','Keep originals until proven new model-eligible records exist')
]
pd.DataFrame(checks,columns=['gate','status','notes']).to_csv(OUTPUT_DIR/'stage13d_decision_gate.csv',index=False)
print(json.dumps(summary,indent=2))
print('\nOutputs:',OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob('*')):print(' -',f.name)

{
  "reviewed_inventory_count": 287,
  "original_ml_eligible_count": 200,
  "coordinate_ineligible_count": 87,
  "review_queue_with_links": 0,
  "review_queue_with_descriptions": 87,
  "independently_approved_coordinate_recoveries": 0,
  "potentially_complete_manual_evidence_packages": 0,
  "model_retraining_authorized": false,
  "current_static_positive_count": 194,
  "current_temporal_positive_count": 121,
  "note": "Manual and independent verification plus original Stage04 spatial/accuracy and full feature QC required before approval."
}

Outputs: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13d_coordinate_recovery
 - coordinate_ineligible_source_rows.csv
 - coordinate_source_review_queue.csv
 - manual_coordinate_evidence_entries.csv
 - proposed_coordinate_evidence_validation.csv
 - stage13d_decision_gate.csv
 - stage13d_summary.json


## How to use the audit

1. Run all cells to produce the evidence queue of excluded records. If the ID gate fails, inspect the source IDs and **fix the join explicitly**—do not use nearest-coordinate matching.
2. Manually investigate the original COOLR/JMG evidence. Only write independently supported corrections into `manual_coordinate_evidence_entries.csv` and run PART D–E again.
3. Entries passing preliminary checks are **not automatically approved**. Independently validate the exact event identity, coordinate evidence, accuracy and Malaysia polygon; then rerun your Stage 04 coordinate-QC and Stage 05 static feature extraction on a separate versioned candidate inventory.
4. If no independently verified corrections are found, stop coordinate recovery, document the exclusions, and **keep your current 194-case static / 121-case temporal models**. If proven corrections exist, reassess complete-case eligibility and the three-algorithm benchmarks on a versioned new dataset.

**Important:** An uncertainty threshold of 5 km was used in the earlier project's audit, but being under that threshold alone does **not** establish location precision appropriate for 30-m slope inference. Perform an additional coordinate-error sensitivity assessment before accepting recovered points into the fine-scale model.